<h1><center><strong> Optuna

In [1]:
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

In [2]:
import optuna
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
import pickle

warnings.filterwarnings('ignore')
warnings.filterwarnings('error')

In [3]:
file_path = '../../../data/data_for_final_models/AgglomerativeClustering_generated_features.csv'
data = pd.read_csv(file_path)

In [4]:
X = data.drop(columns=['Cluster'])
y = data['Cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
def optimize_model(objective_func, timeout=3600 * 2):
    '''Функция для оптимизации гиперпараметров с использованием Optuna

    Параметры:
        objective_func (function): Функция-объектив для оптимизации
        timeout (int): Максимальное время выполнения оптимизации (в секундах)

    Возвращает:
        tuple: Лучшее значение метрики balanced accuracy и лучшие параметры модели
    '''
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_func, timeout=timeout)
    print("Best Balanced Accuracy:", study.best_value)
    print("Best Parameters:", study.best_params)
    return study.best_value, study.best_params

<h1><center><strong> CatBoost

In [6]:
def objective_catboost(trial):
    '''Функция для оптимизации гиперпараметров модели CatBoostClassifier'''
    params = {
        "iterations": 500,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 1, 10),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.05, 1.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
    }

    model = CatBoostClassifier(**params, verbose=0, random_state=42)
    model.fit(X_train, y_train)

    score = model.score(X_test, y_test)
    return score

In [7]:
best_score_cb, best_params_cb = optimize_model(objective_catboost)
print("CatBoost")

[I 2025-01-21 00:23:22,463] A new study created in memory with name: no-name-370f8bd2-fc28-4559-9f13-cd4ca2c8aaca
[I 2025-01-21 00:23:29,258] Trial 0 finished with value: 0.9785879629629629 and parameters: {'learning_rate': 0.00246720252591155, 'depth': 3, 'colsample_bylevel': 0.391275064337521, 'min_data_in_leaf': 10}. Best is trial 0 with value: 0.9785879629629629.
[I 2025-01-21 00:23:56,935] Trial 1 finished with value: 0.9861111111111112 and parameters: {'learning_rate': 0.0028026740226287546, 'depth': 6, 'colsample_bylevel': 0.3315240300186521, 'min_data_in_leaf': 45}. Best is trial 1 with value: 0.9861111111111112.
[I 2025-01-21 00:23:59,765] Trial 2 finished with value: 0.9913194444444444 and parameters: {'learning_rate': 0.07247488358306456, 'depth': 1, 'colsample_bylevel': 0.5195831415577841, 'min_data_in_leaf': 19}. Best is trial 2 with value: 0.9913194444444444.
[I 2025-01-21 00:29:02,656] Trial 3 finished with value: 0.9861111111111112 and parameters: {'learning_rate': 0.00

Best Balanced Accuracy: 0.9953703703703703
Best Parameters: {'learning_rate': 0.09964607298116393, 'depth': 7, 'colsample_bylevel': 0.12589075517182888, 'min_data_in_leaf': 25}
CatBoost


In [8]:
final_model = CatBoostClassifier(**best_params_cb, verbose=0, random_state=42)
final_model.fit(X_train, y_train)

final_model.save_model('../../../models/2_optuna_models/optuna_catboost_model.cb')

<h1><center><strong> LogisticRegression

In [9]:
def objective_logreg(trial):
    '''Функция для оптимизации гиперпараметров модели LogisticRegression'''
    params = {
        "C": trial.suggest_float("C", 0.01, 10.0, log=True),
        "solver": trial.suggest_categorical("solver", ["liblinear", "saga"]),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
        "tol": trial.suggest_float("tol", 0.0001, 0.01, log=True),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False])
    }
    model = LogisticRegression(**params, max_iter=100_000, random_state=42)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    return score

In [10]:
best_score_lr, best_params_lr = optimize_model(objective_logreg)
print("LogisticRegression")

[I 2025-01-21 02:27:30,599] A new study created in memory with name: no-name-b20f79da-c186-4e55-9b09-b2a8f327d3c0
[I 2025-01-21 02:27:33,823] Trial 0 finished with value: 0.9895833333333334 and parameters: {'C': 0.047257207590184355, 'solver': 'liblinear', 'penalty': 'l1', 'tol': 0.00013504285238423124, 'fit_intercept': True}. Best is trial 0 with value: 0.9895833333333334.
[I 2025-01-21 02:27:43,925] Trial 1 finished with value: 0.9913194444444444 and parameters: {'C': 0.024135525576790284, 'solver': 'liblinear', 'penalty': 'l2', 'tol': 0.00014511549428099726, 'fit_intercept': False}. Best is trial 1 with value: 0.9913194444444444.
[I 2025-01-21 02:29:15,475] Trial 2 finished with value: 0.9872685185185185 and parameters: {'C': 0.014917137687633966, 'solver': 'saga', 'penalty': 'l1', 'tol': 0.002737031570237884, 'fit_intercept': True}. Best is trial 1 with value: 0.9913194444444444.
[I 2025-01-21 02:29:19,508] Trial 3 finished with value: 0.9907407407407407 and parameters: {'C': 3.066

Best Balanced Accuracy: 0.9918981481481481
Best Parameters: {'C': 0.02734214269709636, 'solver': 'liblinear', 'penalty': 'l2', 'tol': 0.003331938694596917, 'fit_intercept': False}
LogisticRegression


In [11]:
final_model = LogisticRegression(**best_params_lr, max_iter=100_000, random_state=42)
final_model.fit(X_train, y_train)

with open('../../../models/2_optuna_models/optuna_logistic_regression_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)

<h1><center><strong> RandomForest

In [12]:
def objective_randomforest(trial):
    '''Функция для оптимизации гиперпараметров модели RandomForestClassifier'''
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "max_depth": trial.suggest_int("max_depth", 3, 110),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 4)
    }
    model = RandomForestClassifier(**params, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    return score

In [13]:
best_score_rf, best_params_rf = optimize_model(objective_randomforest)
print("RandomForest")

[I 2025-01-21 04:34:04,469] A new study created in memory with name: no-name-41fdd1ba-9098-4c73-a1f5-0efaddd776af
[I 2025-01-21 04:34:05,251] Trial 0 finished with value: 0.9918981481481481 and parameters: {'n_estimators': 220, 'max_features': 'log2', 'max_depth': 109, 'min_samples_split': 5, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.9918981481481481.
[I 2025-01-21 04:34:52,471] Trial 1 finished with value: 0.9936342592592593 and parameters: {'n_estimators': 477, 'max_features': None, 'max_depth': 41, 'min_samples_split': 10, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.9936342592592593.
[I 2025-01-21 04:34:53,930] Trial 2 finished with value: 0.9947916666666666 and parameters: {'n_estimators': 348, 'max_features': 'sqrt', 'max_depth': 61, 'min_samples_split': 4, 'min_samples_leaf': 2}. Best is trial 2 with value: 0.9947916666666666.
[I 2025-01-21 04:34:55,107] Trial 3 finished with value: 0.9947916666666666 and parameters: {'n_estimators': 283, 'max_features': 'sq

Best Balanced Accuracy: 0.9953703703703703
Best Parameters: {'n_estimators': 215, 'max_features': None, 'max_depth': 43, 'min_samples_split': 4, 'min_samples_leaf': 2}
RandomForest


In [14]:
final_model = RandomForestClassifier(**best_params_rf, n_jobs=-1, random_state=42)
final_model.fit(X_train, y_train)

with open('../../../models/2_optuna_models/optuna_random_forest_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)

<h1><center><strong> ExtraTreesClassifier

In [33]:
def objective_extratrees(trial):
    '''Функция для оптимизации гиперпараметров модели ExtraTreesClassifier'''
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 1, 500),
        "max_depth": trial.suggest_int("max_depth", 1, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 25),
        "criterion": trial.suggest_categorical("criterion", ["entropy", "gini"])
    }
    model = ExtraTreesClassifier(**params, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    return score

In [34]:
best_score_et, best_params_et = optimize_model(objective_extratrees)
print("ExtraTrees")

[I 2025-01-21 10:55:22,480] A new study created in memory with name: no-name-798b9111-f4ad-4cb7-80a9-c251c6206db2
[I 2025-01-21 10:55:22,749] Trial 0 finished with value: 0.9785879629629629 and parameters: {'n_estimators': 33, 'max_depth': 30, 'min_samples_split': 3, 'min_samples_leaf': 7, 'max_leaf_nodes': 25, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9785879629629629.
[I 2025-01-21 10:55:23,560] Trial 1 finished with value: 0.9791666666666666 and parameters: {'n_estimators': 204, 'max_depth': 26, 'min_samples_split': 9, 'min_samples_leaf': 8, 'max_leaf_nodes': 25, 'criterion': 'entropy'}. Best is trial 1 with value: 0.9791666666666666.
[I 2025-01-21 10:55:25,307] Trial 2 finished with value: 0.9791666666666666 and parameters: {'n_estimators': 404, 'max_depth': 27, 'min_samples_split': 9, 'min_samples_leaf': 6, 'max_leaf_nodes': 21, 'criterion': 'entropy'}. Best is trial 1 with value: 0.9791666666666666.
[I 2025-01-21 10:55:25,691] Trial 3 finished with value: 0.979166666

Best Balanced Accuracy: 0.9832175925925926
Best Parameters: {'n_estimators': 10, 'max_depth': 25, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_leaf_nodes': 25, 'criterion': 'entropy'}
ExtraTrees


In [35]:
final_model = ExtraTreesClassifier(**best_params_et, n_jobs=-1, random_state=42)
final_model.fit(X_train, y_train)

with open('../../../models/2_optuna_models/optuna_extra_trees_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)

<h1><center><strong> GradientBoostingClassifier

In [18]:
def objective_gradientboosting(trial):
    '''Функция для оптимизации гиперпараметров модели GradientBoostingClassifier'''
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 700),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0),  
        "max_depth": trial.suggest_int("max_depth", 2, 150), 
        "max_features": trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),  
        "subsample": trial.suggest_float("subsample", 0.1, 1.0),  
        "min_samples_split": trial.suggest_float("min_samples_split", 0.01, 0.5), 
        "min_samples_leaf": trial.suggest_float("min_samples_leaf", 0.01, 0.5),  
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0.0, 0.5),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 150)  
    }
    model = GradientBoostingClassifier(**params, random_state=42)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    return score

In [19]:
best_score_gb, best_params_gb = optimize_model(objective_gradientboosting)
print("GradientBoosting")

[I 2025-01-21 08:34:42,779] A new study created in memory with name: no-name-0741ee79-0748-446e-931b-0224e9a751f2
[I 2025-01-21 08:37:10,876] Trial 0 finished with value: 0.9878472222222222 and parameters: {'n_estimators': 285, 'learning_rate': 0.9430169954897856, 'max_depth': 96, 'max_features': None, 'subsample': 0.5617622249938614, 'min_samples_split': 0.19179471528331074, 'min_samples_leaf': 0.03215786731467181, 'min_impurity_decrease': 0.09571361333287687, 'max_leaf_nodes': 120}. Best is trial 0 with value: 0.9878472222222222.
[I 2025-01-21 08:37:12,147] Trial 1 finished with value: 0.5052083333333334 and parameters: {'n_estimators': 319, 'learning_rate': 0.9367545140754895, 'max_depth': 111, 'max_features': None, 'subsample': 0.20361203208107664, 'min_samples_split': 0.2663261181210885, 'min_samples_leaf': 0.38587500317333534, 'min_impurity_decrease': 0.30759555300170344, 'max_leaf_nodes': 127}. Best is trial 0 with value: 0.9878472222222222.
[I 2025-01-21 08:37:14,851] Trial 2 f

Best Balanced Accuracy: 0.9953703703703703
Best Parameters: {'n_estimators': 612, 'learning_rate': 0.05833915579593037, 'max_depth': 23, 'max_features': 'sqrt', 'subsample': 0.8025852110247288, 'min_samples_split': 0.2626234039365721, 'min_samples_leaf': 0.017261134889094004, 'min_impurity_decrease': 0.025230435303212912, 'max_leaf_nodes': 91}
GradientBoosting


In [20]:
final_model = GradientBoostingClassifier(**best_params_gb, random_state=42)
final_model.fit(X_train, y_train)

with open('../../../models/2_optuna_models/optuna_gradient_boosting_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)